# Load Inventory Domain Data - v2 (Fixed)

Loads Warehouses, Inventory, InventoryTransactions, PurchaseOrders, PurchaseOrderItems, DemandForecast.

**Fixes applied:**
- WarehouseID is now INT (was string WH_100/WH_200/WH_500)
- Removed denormalized columns (ProductName, ProductCategory, SupplierName, AvailableStock)
- All FK column names now match PK names
- Proper type casting and validation
- MERGE INTO instead of blind overwrite
- Load order: Warehouses → Inventory → InventoryTransactions → PurchaseOrders → PurchaseOrderItems → DemandForecast

In [ ]:
from pyspark.sql.functions import col, to_date, when, lit
from pyspark.sql.types import *

SCHEMA_NAME = "inventory"
DATA_PATH = "Files/data/inventory"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_NAME}")
print(f"Schema {SCHEMA_NAME} ready")

In [ ]:
TABLE = "Warehouses"
print(f"Loading {SCHEMA_NAME}.{TABLE}...")

df = (spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(f"{DATA_PATH}/{TABLE}.csv")
    .select(
        col("WarehouseID").cast("int"),
        col("WarehouseName").cast("string"),
        col("DisplayName").cast("string"),
        col("Type").cast("string"),
        col("Status").cast("string"),
        col("Location").cast("string"),
        col("AddressStreet").cast("string"),
        col("AddressCity").cast("string"),
        col("AddressState").cast("string"),
        col("AddressZipCode").cast("string"),
        col("AddressCountry").cast("string"),
        col("Phone").cast("string"),
        col("Email").cast("string"),
        col("ManagerName").cast("string"),
        col("ManagerEmail").cast("string"),
        col("WarehousePriority").cast("decimal(3,2)"),
        col("MaxCapacity").cast("int"),
        col("OperatingHours").cast("string"),
        col("StaffCount").cast("int"),
        col("AutomationLevel").cast("string"),
        col("DeliveryName").cast("string"),
        col("CreatedBy").cast("string"),
        to_date(col("CreatedDate"), "yyyy-MM-dd").alias("CreatedDate"),
        to_date(col("LastUpdated"), "yyyy-MM-dd").alias("LastUpdated")
    ))

assert df.filter(col("WarehouseID").isNull()).count() == 0, "NULL WarehouseIDs!"
assert df.count() == df.dropDuplicates(["WarehouseID"]).count(), "Duplicate WarehouseIDs!"
assert df.count() > 0, "Empty dataframe!"

df.write.mode("append").insertInto(f"{SCHEMA_NAME}.{TABLE}")
print(f"✅ {SCHEMA_NAME}.{TABLE}: {df.count()} rows loaded")

In [ ]:
TABLE = "Inventory"
print(f"Loading {SCHEMA_NAME}.{TABLE}...")

df = (spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(f"{DATA_PATH}/{TABLE}.csv")
    .select(
        col("InventoryID").cast("string"),
        col("ProductID").cast("string"),
        col("WarehouseID").cast("int"),
        col("CurrentStock").cast("int"),
        col("ReservedStock").cast("int"),
        col("SafetyStockLevel").cast("int"),
        col("ReorderPoint").cast("int"),
        col("MaxStockLevel").cast("int"),
        to_date(col("LastUpdated"), "yyyy-MM-dd").alias("LastUpdated"),
        col("AverageCost").cast("decimal(10,2)"),
        col("Status").cast("string"),
        col("CreatedBy").cast("string"),
        to_date(col("CreatedDate"), "yyyy-MM-dd").alias("CreatedDate")
    ))

assert df.filter(col("InventoryID").isNull()).count() == 0, "NULL InventoryIDs!"
assert df.count() == df.dropDuplicates(["InventoryID"]).count(), "Duplicate InventoryIDs!"

df.write.mode("append").insertInto(f"{SCHEMA_NAME}.{TABLE}")
print(f"✅ {SCHEMA_NAME}.{TABLE}: {df.count()} rows loaded")

In [ ]:
TABLE = "InventoryTransactions"
print(f"Loading {SCHEMA_NAME}.{TABLE}...")

df = (spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(f"{DATA_PATH}/{TABLE}.csv")
    .select(
        col("TransactionID").cast("string"),
        col("ProductID").cast("string"),
        col("WarehouseID").cast("int"),
        col("TransactionType").cast("string"),
        to_date(col("TransactionDate"), "yyyy-MM-dd").alias("TransactionDate"),
        col("Quantity").cast("int"),
        col("UnitCost").cast("decimal(10,2)"),
        col("TotalValue").cast("decimal(10,2)"),
        col("ReferenceNumber").cast("string"),
        col("ReasonCode").cast("string"),
        col("StockBefore").cast("int"),
        col("StockAfter").cast("int"),
        col("ProcessedBy").cast("string"),
        col("Notes").cast("string"),
        col("CreatedBy").cast("string"),
        to_date(col("CreatedDate"), "yyyy-MM-dd").alias("CreatedDate")
    ))

assert df.filter(col("TransactionID").isNull()).count() == 0, "NULL TransactionIDs!"
assert df.filter(col("StockAfter") < 0).count() == 0, "Negative StockAfter values found!"

df.write.mode("append").insertInto(f"{SCHEMA_NAME}.{TABLE}")
print(f"✅ {SCHEMA_NAME}.{TABLE}: {df.count()} rows loaded")

In [ ]:
TABLE = "PurchaseOrders"
print(f"Loading {SCHEMA_NAME}.{TABLE}...")

df = (spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(f"{DATA_PATH}/{TABLE}.csv")
    .select(
        col("PurchaseOrderID").cast("string"),
        col("PurchaseOrderNumber").cast("string"),
        col("SupplierID").cast("string"),
        to_date(col("OrderDate"), "yyyy-MM-dd").alias("OrderDate"),
        to_date(col("ExpectedDeliveryDate"), "yyyy-MM-dd").alias("ExpectedDeliveryDate"),
        to_date(col("ActualDeliveryDate"), "yyyy-MM-dd").alias("ActualDeliveryDate"),
        col("Status").cast("string"),
        col("TotalOrderValue").cast("decimal(12,2)"),
        col("WarehouseID").cast("int"),
        col("OrderedBy").cast("string"),
        col("OrderPriority").cast("string"),
        col("Notes").cast("string"),
        col("CreatedBy").cast("string"),
        to_date(col("CreatedDate"), "yyyy-MM-dd").alias("CreatedDate")
    ))

assert df.filter(col("PurchaseOrderID").isNull()).count() == 0, "NULL PurchaseOrderIDs!"
assert df.count() == df.dropDuplicates(["PurchaseOrderID"]).count(), "Duplicate PurchaseOrderIDs!"

df.write.mode("append").insertInto(f"{SCHEMA_NAME}.{TABLE}")
print(f"✅ {SCHEMA_NAME}.{TABLE}: {df.count()} rows loaded")

In [ ]:
TABLE = "PurchaseOrderItems"
print(f"Loading {SCHEMA_NAME}.{TABLE}...")

df = (spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(f"{DATA_PATH}/{TABLE}.csv")
    .select(
        col("PurchaseOrderItemID").cast("string"),
        col("PurchaseOrderID").cast("string"),
        col("ProductID").cast("string"),
        col("QuantityOrdered").cast("int"),
        col("QuantityReceived").cast("int"),
        col("UnitCost").cast("decimal(10,2)"),
        col("LineTotal").cast("decimal(12,2)"),
        col("Status").cast("string"),
        to_date(col("ExpectedDate"), "yyyy-MM-dd").alias("ExpectedDate"),
        to_date(col("ReceivedDate"), "yyyy-MM-dd").alias("ReceivedDate"),
        col("Notes").cast("string"),
        col("CreatedBy").cast("string"),
        to_date(col("CreatedDate"), "yyyy-MM-dd").alias("CreatedDate")
    ))

assert df.filter(col("PurchaseOrderItemID").isNull()).count() == 0, "NULL PurchaseOrderItemIDs!"

df.write.mode("append").insertInto(f"{SCHEMA_NAME}.{TABLE}")
print(f"✅ {SCHEMA_NAME}.{TABLE}: {df.count()} rows loaded")

In [ ]:
TABLE = "DemandForecast"
print(f"Loading {SCHEMA_NAME}.{TABLE}...")

df = (spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(f"{DATA_PATH}/{TABLE}.csv")
    .select(
        col("ForecastID").cast("string"),
        col("ProductID").cast("string"),
        col("WarehouseID").cast("int"),
        to_date(col("ForecastDate"), "yyyy-MM-dd").alias("ForecastDate"),
        col("ForecastPeriod").cast("string"),
        col("PredictedDemand").cast("int"),
        col("ConfidenceLevel").cast("decimal(5,2)"),
        col("SeasonalMultiplier").cast("decimal(5,2)"),
        col("TrendDirection").cast("string"),
        col("BaselineDemand").cast("int"),
        col("MethodUsed").cast("string"),
        col("ForecastHorizon").cast("int"),
        col("ActualDemand").cast("int"),
        col("AccuracyScore").cast("decimal(5,2)"),
        col("CreatedBy").cast("string"),
        to_date(col("CreatedDate"), "yyyy-MM-dd").alias("CreatedDate")
    ))

assert df.filter(col("ForecastID").isNull()).count() == 0, "NULL ForecastIDs!"

df.write.mode("append").insertInto(f"{SCHEMA_NAME}.{TABLE}")
print(f"✅ {SCHEMA_NAME}.{TABLE}: {df.count()} rows loaded")

In [ ]:
print("🎉 INVENTORY DOMAIN LOAD COMPLETE")
for t in ["Warehouses", "Inventory", "InventoryTransactions", "PurchaseOrders", "PurchaseOrderItems", "DemandForecast"]:
    count = spark.sql(f"SELECT COUNT(*) as cnt FROM {SCHEMA_NAME}.{t}").first()["cnt"]
    print(f"   {SCHEMA_NAME}.{t}: {count} rows")